# Hướng 1 — Lọc cộng tác dựa trên sản phẩm

**Người thực hiện: Cường Quốc**  
Dataset: Amazon Reviews 2023 — All Beauty đã làm sạch và ghép metadata.

Notebook xây dựng Item-based Collaborative Filtering (ItemCF), so sánh với Most Popular trên validation và lưu mô hình để chạy lại. Đầu ra là danh sách Top-10/Top-20 sản phẩm chưa được người dùng đánh giá trong lịch sử huấn luyện.

In [1]:
from pathlib import Path
from collections import defaultdict
from time import perf_counter
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.preprocessing import normalize

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data' / 'processed'
REPORT_DIR = PROJECT_DIR / 'reports' / 'item_cf'
MODEL_DIR = PROJECT_DIR / 'models' / 'item_cf'

N_NEIGHBORS = 50
BLOCK_SIZE = 1024
K_VALUES = (10, 20)
CORE_COLUMNS = ['user_id', 'parent_asin', 'rating', 'timestamp']

print('Python:', platform.python_version())
print('pandas:', pd.__version__, '| scipy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('Số sản phẩm lân cận giữ lại:', N_NEIGHBORS)


Python: 3.12.10
pandas: 3.0.6 | scipy: 1.18.1
scikit-learn: 1.9.1
Số sản phẩm lân cận giữ lại: 50


## 1 Đọc các tập đã chia và kiểm tra thời gian

Tập train gồm 70% bản ghi sớm nhất, validation gồm khoảng 20% tiếp theo. Hai mốc đã được chốt ở notebook trước và được kiểm tra lại tại đây. Dữ liệu đầu vào phải có đủ bốn cột lõi và đúng đơn vị timestamp mili giây.


In [2]:
required_files = [DATA_DIR / 'train.parquet', DATA_DIR / 'validation.parquet',
                  DATA_DIR / 'split_config.json']
missing_files = [p.name for p in required_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Thiếu ' + ', '.join(missing_files)
        + '. Đặt notebook cạnh 01_data_preparation.ipynb và chạy lại ô chia tập ở notebook 01.'
    )

split_config = json.loads((DATA_DIR / 'split_config.json').read_text(encoding='utf-8'))
POSITIVE_THRESHOLD = float(split_config['positive_rating_threshold'])
train = pd.read_parquet(DATA_DIR / 'train.parquet', columns=CORE_COLUMNS)
val = pd.read_parquet(DATA_DIR / 'validation.parquet', columns=CORE_COLUMNS)

for name, frame in [('train', train), ('validation', val)]:
    if frame.empty or frame[CORE_COLUMNS].isna().any().any():
        raise ValueError(f'Tập {name} rỗng hoặc có giá trị thiếu trong cột lõi.')
    if not frame['rating'].isin([1, 2, 3, 4, 5]).all():
        raise ValueError(f'Tập {name} có rating không hợp lệ.')
    if frame.duplicated(['user_id', 'parent_asin', 'timestamp']).any():
        raise ValueError(f'Tập {name} có tương tác trùng người dùng, sản phẩm và thời điểm.')

assert train['timestamp'].max() < val['timestamp'].min(), 'Hai tập bị chồng thời gian.'
assert train['timestamp'].max() <= split_config['train_end_timestamp_ms']
assert val['timestamp'].min() > split_config['train_end_timestamp_ms']
assert val['timestamp'].max() <= split_config['validation_end_timestamp_ms']

REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([
    {'Tập': name, 'Số dòng': len(frame), 'Người dùng': frame['user_id'].nunique(),
     'Sản phẩm': frame['parent_asin'].nunique()}
    for name, frame in [('Train', train), ('Validation', val)]
]))


,Tập,Số dòng,Người dùng,Sản phẩm
0,Train,485977,448900,78488
1,Validation,138850,129945,39637


## 2 Chuẩn bị lịch sử và mô hình Most Popular

Một tương tác tích cực là một bản ghi có rating từ 4 sao trở lên. Mỗi cặp người dùng–sản phẩm đóng góp tối đa một lần vào mô hình. Most Popular đếm số người đánh giá tích cực trong train. Toàn bộ sản phẩm đã xuất hiện trong train tạo thành tập ứng viên chung, kể cả những sản phẩm chưa có đánh giá tích cực.

Khi tạo khuyến nghị, cả hai mô hình đều loại sản phẩm người dùng đã đánh giá ở bất kỳ mức sao nào trong train.


In [3]:
popular_started = perf_counter()
train_positive = train[train['rating'] >= POSITIVE_THRESHOLD].drop_duplicates(
    ['user_id', 'parent_asin']
)
if train_positive.empty:
    raise ValueError('Train không có tương tác tích cực để học CF.')

item_ids = pd.Index(sorted(train['parent_asin'].unique()), name='parent_asin')
user_ids = pd.Index(train_positive['user_id'].unique(), name='user_id')
item_to_index = {item_id: i for i, item_id in enumerate(item_ids)}
user_to_index = {user_id: i for i, user_id in enumerate(user_ids)}
warm_users = set(user_ids)
catalog_set = set(item_ids)

seen_by_user = defaultdict(set)
for user_id, item_id in train[['user_id', 'parent_asin']].itertuples(index=False, name=None):
    seen_by_user[user_id].add(item_id)

popularity = (train_positive.groupby('parent_asin').size()
    .reindex(item_ids, fill_value=0).rename('positive_user_count').reset_index()
    .sort_values(['positive_user_count', 'parent_asin'], ascending=[False, True])
    .reset_index(drop=True))
popular_ids = popularity['parent_asin'].tolist()
popular_fit_seconds = perf_counter() - popular_started

def recommend_popular(user_id, k=10, return_info=False):
    if k < 1:
        raise ValueError('k phải lớn hơn hoặc bằng 1.')
    seen = seen_by_user.get(user_id, set())
    result = []
    for item_id in popular_ids:
        if item_id not in seen:
            result.append(item_id)
            if len(result) == k:
                break
    return (result, 0) if return_info else result

print(f'Cặp tích cực trong train: {len(train_positive):,}')
print(f'Người có lịch sử tích cực: {len(user_ids):,}')
display(popularity.head(10))


Cặp tích cực trong train: 356,885
Người có lịch sử tích cực: 332,903


,parent_asin,positive_user_count
0,B085BB7B1M,1552
1,B0BM4GX6TT,1127
2,B00R1TAN7I,973
3,B07C533XCW,922
4,B0107QYW14,765
5,B005BZQHEC,720
6,B09X9BG4FC,708
7,B0C9CWKY9G,699
8,B01M1OFZOG,688
9,B019GBG0IE,667


## 3 Tạo ma trận thưa người dùng và sản phẩm

Ma trận R có giá trị 1 tại các cặp được đánh giá tích cực trong train. Số 0 thể hiện không có tương tác tích cực được ghi nhận, không phải nhãn khẳng định người dùng không thích sản phẩm.

Mỗi sản phẩm được biểu diễn bằng vector các người dùng đã đánh giá tích cực. Chuẩn hóa L2 rồi lấy tích vô hướng cho độ tương tự cosine giữa hai sản phẩm.


In [4]:
cf_started = perf_counter()
user_codes = user_ids.get_indexer(train_positive['user_id'])
item_codes = item_ids.get_indexer(train_positive['parent_asin'])
assert (user_codes >= 0).all() and (item_codes >= 0).all()

R = sparse.csr_matrix(
    (np.ones(len(train_positive), dtype=np.float32), (user_codes, item_codes)),
    shape=(len(user_ids), len(item_ids)), dtype=np.float32,
)
R.sum_duplicates()
assert R.nnz == len(train_positive)
assert np.all(R.data == 1), 'Cần khử trùng cặp user–item trước khi tạo R.'
item_vectors = normalize(R.T.tocsr(), norm='l2', axis=1, copy=True)

def sparse_storage_mb(matrix):
    return (matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes) / 1024**2

print('Kích thước R:', R.shape)
print(f'Số ô được lưu: {R.nnz:,}')
print(f'Dung lượng các mảng CSR của R: {sparse_storage_mb(R):.2f} MiB')


Kích thước R: (332903, 78488)
Số ô được lưu: 356,885
Dung lượng các mảng CSR của R: 3.99 MiB


## 4 Tính độ tương tự theo khối và giữ các sản phẩm lân cận

Với mỗi sản phẩm, giữ tối đa 50 sản phẩm có cosine dương cao nhất và bỏ liên kết với chính nó. Độ tương tự được tính theo từng khối 1.024 sản phẩm bằng phép nhân ma trận thưa. Các trường hợp bằng điểm được sắp theo mã sản phẩm để kết quả ổn định.

50 là cấu hình khởi đầu, chưa được coi là cấu hình tối ưu. Việc cắt số lân cận có thể làm ma trận cuối không đối xứng; mỗi hàng vẫn mô tả các sản phẩm được gợi ý từ một sản phẩm trong lịch sử.


In [5]:
vectors_transpose = item_vectors.T.tocsc()
similarity_indices = []
similarity_values = []
similarity_indptr = [0]

for start in range(0, len(item_ids), BLOCK_SIZE):
    stop = min(start + BLOCK_SIZE, len(item_ids))
    block = (item_vectors[start:stop] @ vectors_transpose).tocsr()
    block.sum_duplicates()

    for local_row in range(stop - start):
        global_row = start + local_row
        a, b = block.indptr[local_row:local_row + 2]
        columns = block.indices[a:b]
        values = block.data[a:b]
        keep = (columns != global_row) & (values > 0)
        columns, values = columns[keep], values[keep]
        order = np.lexsort((columns, -values))[:N_NEIGHBORS]

        if len(order):
            similarity_indices.append(columns[order].astype(np.int32, copy=False))
            similarity_values.append(values[order].astype(np.float32, copy=False))
        similarity_indptr.append(similarity_indptr[-1] + len(order))

    if stop == len(item_ids) or (start // BLOCK_SIZE + 1) % 10 == 0:
        print(f'Đã xử lý {stop:,}/{len(item_ids):,} sản phẩm')

indices = np.concatenate(similarity_indices) if similarity_indices else np.array([], dtype=np.int32)
values = np.concatenate(similarity_values) if similarity_values else np.array([], dtype=np.float32)
S = sparse.csr_matrix(
    (values, indices, np.asarray(similarity_indptr, dtype=np.int64)),
    shape=(len(item_ids), len(item_ids)), dtype=np.float32,
)
S.sort_indices()
S.eliminate_zeros()
cf_fit_seconds = perf_counter() - cf_started

assert np.all(S.diagonal() == 0)
assert np.diff(S.indptr).max(initial=0) <= N_NEIGHBORS
assert np.isfinite(S.data).all()
assert np.all((S.data > 0) & (S.data <= 1.000001))

print(f'Số liên kết lân cận: {S.nnz:,}')
print(f'Sản phẩm có ít nhất một lân cận: {(np.diff(S.indptr) > 0).sum():,}')
print(f'Dung lượng các mảng CSR của S: {sparse_storage_mb(S):.2f} MiB')
print(f'Thời gian xây dựng R và S trong lần chạy này: {cf_fit_seconds:.2f} giây')


Đã xử lý 10,240/78,488 sản phẩm
Đã xử lý 20,480/78,488 sản phẩm
Đã xử lý 30,720/78,488 sản phẩm
Đã xử lý 40,960/78,488 sản phẩm
Đã xử lý 51,200/78,488 sản phẩm
Đã xử lý 61,440/78,488 sản phẩm
Đã xử lý 71,680/78,488 sản phẩm
Đã xử lý 78,488/78,488 sản phẩm
Số liên kết lân cận: 79,734
Sản phẩm có ít nhất một lân cận: 20,148
Dung lượng các mảng CSR của S: 0.91 MiB
Thời gian xây dựng R và S trong lần chạy này: 1.17 giây


## 5 Gợi ý bằng ItemCF và bổ sung bằng Most Popular

Điểm của sản phẩm ứng viên bằng tổng độ tương tự với các sản phẩm tích cực trong lịch sử: `scores = R[user] @ S`. Đây là điểm xếp hạng, không phải xác suất mua hàng hay số sao dự đoán.

Các ứng viên có điểm CF dương được xếp trước. Nếu chưa đủ K sản phẩm hoặc người dùng chưa có lịch sử tích cực, hệ thống bổ sung bằng Most Popular. Vì vậy tên mô hình được báo cáo là **ItemCF + Popular fallback**. Cả hai nguồn đều chỉ dùng thông tin từ train.


In [6]:
def recommend_cf(user_id, k=10, return_info=False):
    if k < 1:
        raise ValueError('k phải lớn hơn hoặc bằng 1.')
    seen = seen_by_user.get(user_id, set())
    result = []
    user_index = user_to_index.get(user_id)

    if user_index is not None:
        scores = (R[user_index:user_index + 1] @ S).tocsr()
        candidate_indices = scores.indices
        candidate_scores = scores.data
        keep = np.array([
            score > 0 and item_ids[index] not in seen
            for index, score in zip(candidate_indices, candidate_scores)
        ], dtype=bool)
        candidate_indices = candidate_indices[keep]
        candidate_scores = candidate_scores[keep]
        order = np.lexsort((candidate_indices, -candidate_scores))[:k]
        result = [item_ids[index] for index in candidate_indices[order]]

    cf_count = len(result)
    if len(result) < k:
        chosen = set(result)
        for item_id in popular_ids:
            if item_id not in seen and item_id not in chosen:
                result.append(item_id)
                chosen.add(item_id)
                if len(result) == k:
                    break

    return (result, cf_count) if return_info else result

demo_user = train_positive['user_id'].value_counts().index[0]
demo_ids, demo_cf_count = recommend_cf(demo_user, k=20, return_info=True)
assert len(demo_ids) == len(set(demo_ids))
assert set(demo_ids).isdisjoint(seen_by_user[demo_user])
assert set(demo_ids).issubset(catalog_set)

demo_result = pd.DataFrame({
    'rank': range(1, len(demo_ids) + 1),
    'parent_asin': demo_ids,
    'source': ['ItemCF' if i < demo_cf_count else 'Most Popular'
               for i in range(len(demo_ids))],
})
print('Người dùng minh họa:', demo_user)
print(f'{demo_cf_count}/{len(demo_ids)} sản phẩm đến từ điểm CF dương.')
display(demo_result)
demo_result.to_csv(REPORT_DIR / 'cf_demo_top20.csv', index=False, encoding='utf-8-sig')


Người dùng minh họa: AG73BVBKUOH22USSFJA5ZWL7AKXA
20/20 sản phẩm đến từ điểm CF dương.


,rank,parent_asin,source
0,1,B07KMTCM4F,ItemCF
1,2,B07LGMF5ZX,ItemCF
2,3,B07QK32FWP,ItemCF
3,4,B081RBRCMJ,ItemCF
4,5,B085HHFKVK,ItemCF
5,6,B0868XK2BV,ItemCF
6,7,B07SD7GP25,ItemCF
7,8,B07DPYSP3X,ItemCF
8,9,B081G2H27P,ItemCF
9,10,B07R1HNFZX,ItemCF


## 6 Tạo đáp án validation và thống kê mức thiếu lịch sử

Đáp án gồm các cặp user–item có rating từ 4 sao trong validation. Mỗi cặp chỉ xuất hiện một lần và các sản phẩm đã được chính người đó đánh giá trong train được loại khỏi đáp án vì mục tiêu là khám phá sản phẩm khác.

Giữ cả sản phẩm đáp án chưa xuất hiện trong train; chúng sẽ không thể được hai mô hình hiện tại gợi ý và được tính là không trúng. Báo cáo riêng nhóm có ít nhất một tương tác tích cực trong train để xem rõ phần đóng góp của cá nhân hóa. Người dùng không trúng sản phẩm nào vẫn được tính trong mẫu số.


In [7]:
val_positive = (val[val['rating'] >= POSITIVE_THRESHOLD]
                [['user_id', 'parent_asin']].drop_duplicates())
truth_by_user = defaultdict(set)
excluded_seen_pairs = 0
for user_id, item_id in val_positive.itertuples(index=False, name=None):
    if item_id in seen_by_user.get(user_id, set()):
        excluded_seen_pairs += 1
    else:
        truth_by_user[user_id].add(item_id)
if not truth_by_user:
    raise ValueError('Validation không còn đáp án sau khi áp dụng quy tắc đánh giá.')

validation_diagnostics = {
    'users_with_ground_truth': len(truth_by_user),
    'users_with_positive_training_history': sum(user in warm_users for user in truth_by_user),
    'novel_positive_pairs': sum(len(items) for items in truth_by_user.values()),
    'excluded_previously_reviewed_pairs': excluded_seen_pairs,
    'positive_pairs_with_items_unseen_in_train': sum(
        item not in catalog_set for items in truth_by_user.values() for item in items
    ),
}
display(pd.DataFrame(validation_diagnostics.items(), columns=['Chỉ tiêu', 'Kết quả']))


,Chỉ tiêu,Kết quả
0,users_with_ground_truth,85951
1,users_with_positive_training_history,4392
2,novel_positive_pairs,91672
3,excluded_previously_reviewed_pairs,18
4,positive_pairs_with_items_unseen_in_train,41801


## 7 So sánh Most Popular với ItemCF trên cùng validation

Precision@K = số sản phẩm trúng / K. Recall@K = số sản phẩm trúng / số sản phẩm trong đáp án của người dùng. NDCG@K tăng khi sản phẩm trúng đứng gần đầu danh sách. Lấy trung bình theo người dùng.

Catalog Coverage@K là tỷ lệ sản phẩm trong danh mục train xuất hiện ít nhất một lần trong các danh sách gợi ý của nhóm đang đánh giá. Độ phủ đo khả năng phân phối gợi ý trong danh mục, chưa chứng minh công bằng giữa các nhà bán hàng.

Cả hai mô hình trả về cùng số lượng gợi ý với cùng danh mục ứng viên và quy tắc lọc. Các số đo này đánh giá xếp hạng review tích cực, không phải mức tăng doanh số.


In [8]:
evaluation_started = perf_counter()
discounts = 1 / np.log2(np.arange(2, max(K_VALUES) + 2))
models = {'Most Popular': recommend_popular, 'ItemCF + Popular fallback': recommend_cf}
metric_rows = []
coverage = defaultdict(set)
inference_times = {}

for model_name, recommender in models.items():
    model_started = perf_counter()
    for position, (user_id, relevant_items) in enumerate(truth_by_user.items(), start=1):
        recommendations, cf_count = recommender(
            user_id, k=max(K_VALUES), return_info=True)
        assert len(recommendations) == max(K_VALUES)
        assert len(recommendations) == len(set(recommendations))
        assert set(recommendations).isdisjoint(seen_by_user.get(user_id, set()))
        is_warm = user_id in warm_users

        for k in K_VALUES:
            top_k = recommendations[:k]
            hits = np.array([item in relevant_items for item in top_k], dtype=float)
            count = hits.sum()
            ideal_dcg = discounts[:min(k, len(relevant_items))].sum()
            metric_rows.append({
                'model': model_name, 'user_id': user_id, 'k': k,
                'has_positive_history': is_warm,
                'precision': count / k,
                'recall': count / len(relevant_items),
                'ndcg': (hits * discounts[:len(hits)]).sum() / ideal_dcg,
                'cf_count': min(cf_count, k),
            })
            coverage[(model_name, k, 'Toàn bộ')].update(top_k)
            if is_warm:
                coverage[(model_name, k, 'Có lịch sử tích cực')].update(top_k)

        if position % 25000 == 0:
            print(f'{model_name}: {position:,}/{len(truth_by_user):,} người dùng')
    inference_times[model_name] = perf_counter() - model_started

per_user_metrics = pd.DataFrame(metric_rows)
result_rows = []
for model_name in models:
    for k in K_VALUES:
        selected = per_user_metrics[
            (per_user_metrics['model'] == model_name) & (per_user_metrics['k'] == k)]
        groups = {'Toàn bộ': selected,
                  'Có lịch sử tích cực': selected[selected['has_positive_history']]}
        for group_name, group in groups.items():
            result_rows.append({
                'model': model_name, 'k': k, 'group': group_name,
                'users': len(group),
                'Precision@K': group['precision'].mean(),
                'Recall@K': group['recall'].mean(),
                'NDCG@K': group['ndcg'].mean(),
                'CatalogCoverage@K': len(coverage[(model_name, k, group_name)]) / len(item_ids),
                'CF_slot_fraction': (group['cf_count'] / k).mean(),
                'popular_only_list_fraction': group['cf_count'].eq(0).mean(),
            })

results = pd.DataFrame(result_rows)
results.to_csv(REPORT_DIR / 'validation_comparison.csv', index=False, encoding='utf-8-sig')
per_user_metrics.to_csv(REPORT_DIR / 'validation_per_user.csv.gz', index=False, compression='gzip')
print('Kết quả validation; các chỉ số dưới đây ở thang 0–1:')
display(results[['model', 'k', 'group', 'users', 'Precision@K', 'Recall@K',
                 'NDCG@K', 'CatalogCoverage@K']].round(6))

print('Mức sử dụng CF và mức phải dùng hoàn toàn Most Popular:')
display(results[results['model'] == 'ItemCF + Popular fallback'][
    ['k', 'group', 'users', 'CF_slot_fraction', 'popular_only_list_fraction']].round(6))
print(f'Thời gian toàn bộ vòng đánh giá: {perf_counter() - evaluation_started:.2f} giây')


Most Popular: 25,000/85,951 người dùng
Most Popular: 50,000/85,951 người dùng
Most Popular: 75,000/85,951 người dùng
ItemCF + Popular fallback: 25,000/85,951 người dùng
ItemCF + Popular fallback: 50,000/85,951 người dùng
ItemCF + Popular fallback: 75,000/85,951 người dùng
Kết quả validation; các chỉ số dưới đây ở thang 0–1:


,model,k,group,users,Precision@K,Recall@K,NDCG@K,CatalogCoverage@K
0,Most Popular,10,Toàn bộ,85951,0.001437,0.014173,0.006655,0.000140
1,Most Popular,10,Có lịch sử tích cực,4392,0.000888,0.008500,0.004118,0.000140
2,Most Popular,20,Toàn bộ,85951,0.000971,0.019158,0.007921,0.000268
3,Most Popular,20,Có lịch sử tích cực,4392,0.000649,0.012485,0.005147,0.000268
4,ItemCF + Popular fallback,10,Toàn bộ,85951,0.001437,0.014132,0.006657,0.090116
5,ItemCF + Popular fallback,10,Có lịch sử tích cực,4392,0.000888,0.007698,0.004159,0.090116
6,ItemCF + Popular fallback,20,Toàn bộ,85951,0.000976,0.019167,0.007935,0.102793
7,ItemCF + Popular fallback,20,Có lịch sử tích cực,4392,0.000740,0.012662,0.005422,0.102793


Mức sử dụng CF và mức phải dùng hoàn toàn Most Popular:


,k,group,users,CF_slot_fraction,popular_only_list_fraction
4,10,Toàn bộ,85951,0.022196,0.965189
5,10,Có lịch sử tích cực,4392,0.434381,0.318761
6,20,Toàn bộ,85951,0.017665,0.965189
7,20,Có lịch sử tích cực,4392,0.345697,0.318761


Thời gian toàn bộ vòng đánh giá: 4.11 giây


## 8 Lưu mô hình và dữ liệu bàn giao

Các ma trận `.npz` đi cùng bảng ánh xạ người dùng/sản phẩm và thông tin cấu hình. Khi phục vụ lại khuyến nghị, cần dùng cùng lịch sử train để loại các sản phẩm đã đánh giá. Phần ma trận được lưu ở dạng thưa.

Thời gian trong tệp cấu hình là thời gian quan sát trên máy đang chạy, có thể khác giữa các máy. `model_build_seconds` đo phần chuẩn bị Most Popular và phần tạo R/S của CF, không gồm đọc dữ liệu. `evaluation_seconds_by_model` đo cả gợi ý, tính chỉ số và kiểm tra danh sách trong vòng lặp đánh giá.


In [9]:
sparse.save_npz(MODEL_DIR / 'user_item_train.npz', R)
sparse.save_npz(MODEL_DIR / 'item_neighbors.npz', S)
pd.DataFrame({'item_index': np.arange(len(item_ids)), 'parent_asin': item_ids}).to_csv(
    MODEL_DIR / 'items.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'user_index': np.arange(len(user_ids)), 'user_id': user_ids}).to_csv(
    MODEL_DIR / 'users.csv', index=False, encoding='utf-8-sig')
popularity.to_csv(MODEL_DIR / 'popularity.csv', index=False, encoding='utf-8-sig')

train_fingerprint = hashlib.sha256(
    pd.util.hash_pandas_object(train[CORE_COLUMNS], index=False).values.tobytes()
).hexdigest()
model_config = {
    'model': 'binary_item_cosine_cf_with_popular_fallback',
    'n_neighbors': N_NEIGHBORS, 'block_size': BLOCK_SIZE,
    'positive_rating_threshold': POSITIVE_THRESHOLD,
    'k_values': list(K_VALUES), 'training_rows': len(train),
    'positive_training_pairs': len(train_positive),
    'training_fingerprint': train_fingerprint,
    'split_config': split_config,
    'validation_diagnostics': validation_diagnostics,
    'model_build_seconds': {'Most Popular': popular_fit_seconds, 'ItemCF_R_and_S': cf_fit_seconds},
    'evaluation_seconds_by_model': inference_times,
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'numpy': np.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__},
}
(MODEL_DIR / 'model_config.json').write_text(
    json.dumps(model_config, ensure_ascii=False, indent=2), encoding='utf-8')
(REPORT_DIR / 'validation_diagnostics.json').write_text(
    json.dumps(validation_diagnostics, ensure_ascii=False, indent=2), encoding='utf-8')

print('Đã lưu kết quả vào reports/item_cf và mô hình vào models/item_cf.')
print('Bảng so sánh: validation_comparison.csv')
print('Danh sách gợi ý minh họa: cf_demo_top20.csv')


Đã lưu kết quả vào reports/item_cf và mô hình vào models/item_cf.
Bảng so sánh: validation_comparison.csv
Danh sách gợi ý minh họa: cf_demo_top20.csv
